# Etapa 4: Feature Engineering y Creación de Capa Gold

Este cuaderno realiza la construcción de la matriz analítica de candidatos para el recomendador binario. Excluye productos que el cliente ya posee activamente, calcula variables agregadas financieras y simula el target predictivo bancario balanceado y reproducible.

In [1]:
import sys
import os
from pathlib import Path

# Resolver la ruta raíz del proyecto de forma dinámica y portable
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyarrow", "pandas", "scikit-learn", "joblib", "numpy"])
    print("Dependencias instaladas en Colab.")
    
    # Intentar montar Google Drive automáticamente si no está montado
    if not Path('/content/drive').exists():
        try:
            from google.colab import drive
            drive.mount('/content/drive')
        except Exception as e:
            print("No se pudo montar Drive automáticamente. Por favor, móntelo en el panel izquierdo de Colab.")

current_dir = Path(os.getcwd()).resolve()
if IN_COLAB:
    # Rutas de búsqueda comunes en Google Drive y Colab
    possible_paths = [
        Path('/content/drive/MyDrive/Colab Notebooks/Proyecto'),
        Path('/content/drive/MyDrive/Proyecto'),
        Path('/content/Proyecto'),
        Path('/content')
    ]
    for p in possible_paths:
        if (p / "notebooks").exists():
            current_dir = p / "notebooks"
            break

if current_dir.name == "notebooks":
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

## 1. Ejecutar Construcción de Capa Gold

In [2]:
from src.feature_engineering import FeatureEngineer

engineer = FeatureEngineer(PROJECT_ROOT)
engineer.silver_to_gold()

[Gold] Iniciando cruces de Feature Engineering...
[Gold] Matriz analítica de candidatos construida y guardada en la capa Gold exitosamente.


## 2. Validación de Dimensiones Resultantes
Inspeccionamos los archivos generados en la capa Gold para entrenamiento y predicción.

In [3]:
import pandas as pd

gold_dir = PROJECT_ROOT / "data" / "gold"
train_df = pd.read_parquet(gold_dir / "dataset_entrenamiento_nbp.parquet")
pred_df = pd.read_parquet(gold_dir / "dataset_prediccion_nbp.parquet")

print("Estructura del Dataset de Entrenamiento:")
print(f"Filas: {train_df.shape[0]} | Columnas: {train_df.shape[1]}")
print("Distribución de la Variable Objetivo (y):")
print(train_df["y"].value_counts(normalize=True))

print("\nEstructura del Dataset de Predicción:")
print(f"Filas: {pred_df.shape[0]} | Columnas: {pred_df.shape[1]} (y debe ser Nula)")
print(f"Nulos en y: {pred_df['y'].isnull().sum()} / {pred_df.shape[0]}")

Estructura del Dataset de Entrenamiento:
Filas: 14947 | Columnas: 18
Distribución de la Variable Objetivo (y):
y
0    0.896635
1    0.103365
Name: proportion, dtype: float64

Estructura del Dataset de Predicción:
Filas: 16000 | Columnas: 18 (y debe ser Nula)
Nulos en y: 16000 / 16000
